## Init

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
%run /Workspace/Users/axl.dxn@gmail.com/atlikon_pipeline/1_setup/utilities

## Configure widgets

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(f"catalog: {catalog}, data_source: {data_source}")

## Read from customers bronze table 

In [0]:
df_bronze = (
    spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
)

## Remove duplicate data

In [0]:
df_duplicates = (
    df_bronze
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

display(df_duplicates)

In [0]:
print("Rows before dropping duplicates: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows after dropping duplicates: ", df_silver.count())

## Handling leading and trailing spaces in string values

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = (
    df_silver
    .withColumn("customer_name", F.trim(F.col("customer_name")))
)

## Spelling errors in city names

In [0]:
df_silver.select("city").distinct().show()

In [0]:
city_mapping = {
    "Bengaluruu" : "Bengaluru",
    "Bengalore" : "Bengaluru",

    "Hyderabadd" : "Hyderabad",
    "Hyderbad" : "Hyderabad",

    "NewDelhi" : "New Delhi",
    "NewDheli" : "New Delhi",
    "NewDelhee" : "New Delhi"  
}

allowed = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset = ["city"])
    .withColumn(
        "city", 
        F.when(F.col("city").isNull(), None)
         .when(F.col("city").isin(allowed), F.col("city"))
         .otherwise(None)
    )
)

df_silver.select("city").distinct().show()

## Title case fix in customer names

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer_name", 
        F.when(F.col("customer_name").isNull(), None)
         .otherwise(F.initcap("customer_name"))
    )
)

df_silver.select("customer_name").distinct().show()

## Handling NULL in city 

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customers_names = ["Sprintx Nutrition", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(null_customers_names)).show(truncate=False)

In [0]:
# Simulation of city corrections confirmed by the business team.

customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix_null = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix_null)

In [0]:
df_silver = (
    df_silver
    .join(df_fix_null, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city", "fixed_city")
    )
    .drop("fixed_city")
)

display(df_silver.orderBy("customer_id"))

## Change data type of customer_id

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))

print(df_silver.printSchema())

## Transforming DF to match the format of `dim_customers` table in the gold layer

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer", 
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

display(df_silver.limit(10))

## Write in silver table 

In [0]:
(
    df_silver
    .write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("mergeSchema", "true")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
)

## Sanity check of silver table

In [0]:
query = f"SELECT * FROM {catalog}.{silver_schema}.{data_source} LIMIT 10;"

df_check = spark.sql(query)

display(df_check.limit(10))